# The jet wake at hadron level

The wake notebook (`jet_wake.ipynb`) shows $\Delta e = e(\text{jet}) - e(\text{no jet})$ in the
fluid. This one asks the experimental question: **does that deposit survive particlization,
and what does it look like in the hadrons?**

Two runs of central Au+Au 200 GeV on the **same events**, both viscous (Israel–Stewart,
$\eta/s = 0.08$), both particlized by iSS at $T_\text{sw} = 0.15$ GeV. The medium is
normalized to the measured $dN_{ch}/d\eta \approx 650$ (§2 shows how close it gets):

| | what is particlized | file |
|---|---|---|
| **jet** | IC → jet (Matter + LBT) → CausalLiquefier → `FastHydro_jet` → iSS | `hadrons_jet.npz` |
| **background** | IC → `FastHydro_bg` → iSS | `hadrons_bg.npz` |

The wake is **jet − background, event by event**. Event *k* of the two runs has the same initial
condition, because the IC depends only on `(run.seed, event index)`; §1 checks the hashes.
Every quantity is an average over iSS oversamples. Its error is the compound-Poisson one,
$\mathrm{Var} = \sum_i w_i^2 / N_\text{os}^2$, which is exact for iSS's independent sampling
(`fasthydro/hadrons.py`).

Make the files with **`example/make_hadron_wake_data.py`** (§1 prints the command). By
default the hard process is **PGun**: one 60 GeV parton that starts at the fireball centre and
moves along $+x$ at $y = 0$ in every event. Every wake therefore sits in the same place, and
the events stack.

**What is not in here.**
- The particlization is ideal. The fluid is viscous, but FastHydro stores
  $\pi^{\mu\nu} = \Pi = 0$ (fast_data's frames carry only $e$ and $v$), so iSS samples without
  $\delta f$.
- There is no hadronic afterburner (SMASH is off in this build). iSS decays the resonances,
  which it does only for the UrQMD list, so the fluid ends on MUSIC's EOS 9 (`hotqcd`).
- The jet's own surviving partons are not hadronized: these are the **soft** hadrons only,
  i.e. the medium and its response.

In [ ]:
import json, os, sys
import numpy as np
import matplotlib.pyplot as plt

for p in (os.path.join("..", "python"), os.path.join(os.getcwd(), "..", "python")):
    if os.path.isdir(p) and p not in sys.path:
        sys.path.insert(0, p)
from fasthydro.hadrons import load, SPECIES

OUT = os.environ.get("HADRON_WAKE_OUT", "out_hadron_wake")

# --- one fixed visual language ---------------------------------------------------------------
# Hues are ASSIGNED to entities, never cycled, and every series also has its own line style or
# marker, so identity never rests on colour alone. Checked with the dataviz palette validator
# (light surface, deutan/protan/tritan): all pass.
C_BG, C_JET, C_DIFF = "#0072B2", "#D55E00", "#222222"
LEG = {"bg":  dict(color=C_BG,  ls="-",  lw=1.8, label="background"),
       "jet": dict(color=C_JET, ls="--", lw=1.8, label="jet")}
SP = {"pi": dict(color="#0072B2", marker="o", ls="-",  label=r"$\pi^\pm$"),
      "K":  dict(color="#D55E00", marker="s", ls="--", label=r"$K^\pm$"),
      "p+pbar": dict(color="#009E73", marker="^", ls=":", label=r"$p+\bar p$")}
CMAP_DIFF = "RdBu_r"     # diverging about a neutral 0, for signed differences only

def tidy(ax, xlabel=None, ylabel=None, title=None):
    ax.grid(True, alpha=0.25, lw=0.6); ax.set_axisbelow(True)
    for s in ("top", "right"):
        ax.spines[s].set_visible(False)
    if xlabel: ax.set_xlabel(xlabel)
    if ylabel: ax.set_ylabel(ylabel)
    if title: ax.set_title(title, loc="left", fontsize=11)
    return ax

def band(ax, x, y, e, **kw):
    # a line with its 1-sigma band: the band in the line's own colour, quieter
    ln, = ax.plot(x, y, **kw)
    ax.fill_between(x, y - e, y + e, color=ln.get_color(), alpha=0.18, lw=0)
    return ln

def centers(e):
    return 0.5 * (e[1:] + e[:-1])

plt.rcParams.update({"figure.dpi": 110, "font.size": 10, "axes.titlesize": 11})

## 1. The data and the controls

Nothing below means anything unless, for every event used:

1. **both runs started from the same initial condition** (same IC sha256), otherwise
   jet − bg is not the jet;
2. **both freeze-out surfaces closed** in τ and x, y inside the stored evolution. The η edges
   are open by construction, so the analysis stays at mid-rapidity.

Events that fail either are dropped here, and nowhere else.

In [ ]:
LEGS = ("jet", "bg")
paths = {l: (os.path.join(OUT, f"hadrons_{l}.npz"), os.path.join(OUT, f"{l}_events.json"))
         for l in LEGS}
missing = [p for l in LEGS for p in paths[l] if not os.path.exists(p)]
if missing:
    print("Missing:\n  " + "\n  ".join(missing))
    print("\nGenerate both runs from your X-SCAPE build tree:\n")
    print("    python ../external_packages/js-contrib/contribs/FastHydro/example/"
          "make_hadron_wake_data.py\n")
    print("then run this notebook from that build tree, or point it at the output:\n")
    print("    HADRON_WAKE_OUT=<build>/out_hadron_wake jupyter lab hadron_wake.ipynb")
    raise SystemExit("generate the files above, then re-run this cell")

META = {l: json.load(open(paths[l][1])) for l in LEGS}
H = {l: load(paths[l][0], META[l]["n_oversample"]) for l in LEGS}

rows, good = [], []
for k in range(min(H["jet"].n_events, H["bg"].n_events,
                   len(META["jet"]["events"]), len(META["bg"]["events"]))):
    ej, eb = META["jet"]["events"][k], META["bg"]["events"][k]
    same = ej["ic_sha256"] == eb["ic_sha256"]
    closed = ej["closure"]["closed"] and eb["closure"]["closed"]
    P = np.asarray(ej.get("deposited_P") or [0, 0, 0, 0])
    rows.append((k, same, closed, ej.get("n_droplets", 0), *P))
    if same and closed:
        good.append(k)

print(f"jet: {H['jet'].n_events} events x {H['jet'].n_os} oversamples;  "
      f"bg: {H['bg'].n_events} events x {H['bg'].n_os} oversamples\n")
print(" ev  same IC  closed  droplets   E_dep   px_dep   py_dep   pz_dep  [GeV]")
for r in rows:
    print(f"{r[0]:3d}  {str(r[1]):7s}  {str(r[2]):6s}  {r[3]:8d}  {r[4]:6.2f}  {r[5]:7.2f}  "
          f"{r[6]:7.2f}  {r[7]:7.2f}")
print(f"\nusing {len(good)} of {len(rows)} events: {good}")
assert good, "no usable event pair"

### The jet axis

Each event's axis is its **leading shower initiator**: the highest-energy parton that
Matter/LBT started from. For PGun that is the one parton, at $\phi = 0$, $\eta = 0$. For
PythiaGun it is the harder jet of the dijet, and the recoiling jet's own response then sits
on the away side, on top of the near-side jet's diffusion wake. From here on, every hadron
carries $\Delta\phi = \phi - \phi_J$ and $\Delta\eta = \eta - \eta_J$ of its own event.

In [ ]:
def jet_axis(ev):
    e = META["jet"]["events"][ev]
    if e.get("initiators"):
        cols = e["initiator_columns"]
        ini = np.asarray(e["initiators"])
        lead = ini[np.argmax(ini[:, cols.index("E")])]
        px, py, pz = (lead[cols.index(c)] for c in ("px", "py", "pz"))
    else:                                   # fall back to the direction of what was deposited
        _, px, py, pz = e["deposited_P"]
    pt = np.hypot(px, py)
    return np.arctan2(py, px), np.arcsinh(pz / pt), pt

AXIS = {k: jet_axis(k) for k in good}
for k, (phi, eta, pt) in AXIS.items():
    print(f"event {k}: leading initiator p_T = {pt:6.1f} GeV at phi = {phi:+.2f}, eta = {eta:+.2f}")

for l in LEGS:
    h = H[l]
    phiJ = np.array([AXIS.get(k, (0, 0, 0))[0] for k in range(h.n_events)])[h.event]
    etaJ = np.array([AXIS.get(k, (0, 0, 0))[1] for k in range(h.n_events)])[h.event]
    h.dphi = np.angle(np.exp(1j * (h.phi - phiJ)))     # (-pi, pi]
    h.deta = h.eta - etaJ
    h.p_along = h.px * np.cos(phiJ) + h.py * np.sin(phiJ)   # transverse momentum along the jet

EV = good

## 2. The hadrons: background and jet events

The bulk first, since the wake is measured against it. All spectra are per event and per
oversample. The two legs are drawn together; the tiny wake is invisible at this scale, which
is the point of §3.

In [ ]:
Y = 0.5                                          # |y| window for spectra and yields
print(f"dN/dy at |y| < {Y}, per event (background | jet - background)")
for name in ("pi+", "pi-", "K+", "K-", "p", "pbar"):
    vals = []
    for l in LEGS:
        h = H[l]
        m = h.species(name) & (np.abs(h.y) < Y)
        vals.append(h.total(m, events=EV))
    (nj, sj), (nb, sb) = vals
    print(f"  {name:5s} {nb / (2 * Y):8.2f} +- {sb / (2 * Y):5.2f}  |  "
          f"{(nj - nb) / (2 * Y):+6.3f} +- {np.hypot(sj, sb) / (2 * Y):5.3f}")
h = H["bg"]
nch = h.total(h.charged & (np.abs(h.eta) < Y), events=EV)[0] / (2 * Y)
print(f"\n  dN_ch/deta |eta| < {Y}: {nch:.1f}")

In [ ]:
pt_edges = np.linspace(0.0, 3.0, 31)
ptc = centers(pt_edges)
fig, axs = plt.subplots(1, 3, figsize=(13, 4.0), sharey=True)
for ax, name in zip(axs, ("pi", "K", "p+pbar")):
    for l in LEGS:
        h = H[l]
        m = h.species(name) & (np.abs(h.y) < Y)
        n, e = h.hist(h.pt, pt_edges, m, events=EV)
        norm = 2 * np.pi * ptc * np.diff(pt_edges) * (2 * Y)
        ax.errorbar(ptc, n / norm, e / norm, **LEG[l], capsize=0, elinewidth=0.8)
    ax.set_yscale("log")
    tidy(ax, r"$p_T$ [GeV]", r"$\frac{1}{2\pi p_T}\frac{dN}{dp_T\,dy}$ [GeV$^{-2}$]" if ax is axs[0] else None,
         SP[name]["label"] + f",  $|y|<{Y}$")
axs[0].legend(frameon=False)
fig.suptitle("Identified-hadron spectra", x=0.01, ha="left", fontsize=12)
fig.tight_layout()

In [ ]:
eta_edges = np.linspace(-4.0, 4.0, 33)
etc = centers(eta_edges)
fig, axs = plt.subplots(1, 2, figsize=(12, 4.0))

ax = axs[0]
for l in LEGS:
    h = H[l]
    n, e = h.hist(h.eta, eta_edges, h.charged, events=EV)
    band(ax, etc, n / np.diff(eta_edges), e / np.diff(eta_edges), **LEG[l])
tidy(ax, r"$\eta$", r"$dN_{ch}/d\eta$", "Charged-hadron pseudorapidity density")
ax.set_xlim(eta_edges[0], eta_edges[-1])
ax.legend(frameon=False, loc="lower center")

ax = axs[1]
h = H["bg"]
for name in ("pi", "K", "p+pbar"):
    m = h.species(name)
    s, _ = h.hist(h.eta, eta_edges, m, weights=h.pt, events=EV)
    n, _ = h.hist(h.eta, eta_edges, m, events=EV)
    mean = np.divide(s, n, out=np.full_like(s, np.nan), where=n > 0)
    # error on a mean of n Poisson-sampled values: sigma_pT / sqrt(N_total)
    s2, _ = h.hist(h.eta, eta_edges, m, weights=h.pt ** 2, events=EV)
    ntot = n * h.n_os * len(EV)
    err = np.sqrt(np.clip(s2 / np.maximum(n, 1e-30) - mean ** 2, 0, None) / np.maximum(ntot, 1))
    ax.errorbar(etc, mean, err, **SP[name], ms=4, lw=1.4, capsize=0)
tidy(ax, r"$\eta$", r"$\langle p_T \rangle$ [GeV]", r"$\langle p_T\rangle$ vs $\eta$, background")
ax.legend(frameon=False, ncol=3, loc="lower center")
fig.tight_layout()

In [ ]:
phi_edges = np.linspace(-np.pi, np.pi, 37)
phc = centers(phi_edges)
fig, axs = plt.subplots(1, 2, figsize=(12, 3.8))
ax = axs[0]
for l in LEGS:
    h = H[l]
    m = h.charged & (np.abs(h.eta) < 1)
    n, e = h.hist(h.phi, phi_edges, m, events=EV)
    band(ax, phc, n / np.diff(phi_edges), e / np.diff(phi_edges), **LEG[l])
tidy(ax, r"$\phi$ (lab)", r"$dN_{ch}/d\phi$", r"Azimuth in the lab, $|\eta|<1$")
ax.legend(frameon=False)
ax.set_xlim(-np.pi, np.pi)

ax = axs[1]
for k in EV:
    h = H["bg"]
    m = h.charged & (np.abs(h.eta) < 1)
    n, _ = h.hist(h.phi, phi_edges, m, events=[k])
    ax.plot(phc, n / n.mean(), lw=1.0, alpha=0.8, color=C_BG)
tidy(ax, r"$\phi$ (lab)", "per-event shape (norm. to mean)",
     "Background, event by event: the fluctuating\ninitial geometry shows through")
ax.set_xlim(-np.pi, np.pi)
fig.tight_layout()

## 3. The wake: jet − background

From here on everything is the **difference** of the two legs, event by event, relative to
each event's jet axis:

$$\Delta X = \langle X \rangle_\text{jet} - \langle X \rangle_\text{bg}, \qquad
\sigma^2_{\Delta X} = \sigma^2_\text{jet} + \sigma^2_\text{bg}.$$

What a hydrodynamic wake predicts for a jet moving along $\Delta\phi = 0$:

- a **near-side excess** of soft hadrons: the energy and momentum the jet deposited, carried by
  the Mach front and the flow it drives, emitted around the jet direction and spread well
  beyond the jet cone;
- a **depletion on the away side** ($\Delta\phi \approx \pi$), the **diffusion wake**: the
  deposited momentum drags matter along the jet, which leaves less flow, and fewer hadrons,
  in the opposite direction.

Where the excess sits in $p_T$ is itself a measurement. Matter that was only heated would
look like the bulk. Matter that the deposit set moving along the jet is flow-boosted, so its
spectrum is harder and richer in heavy hadrons, as radial flow makes the bulk's
$\langle p_T\rangle$ rise with mass.

In [ ]:
def delta(hfun):
    # hfun(h) -> (value, err) for one leg; -> (jet - bg, error)
    (j, sj), (b, sb) = hfun(H["jet"]), hfun(H["bg"])
    return j - b, np.sqrt(sj ** 2 + sb ** 2)

MID = 1.0          # |Delta eta| window for the azimuthal analysis
dphi_edges = np.linspace(-np.pi, np.pi, 25)
dpc = centers(dphi_edges)
dw = np.diff(dphi_edges)

fig, axs = plt.subplots(1, 2, figsize=(12.5, 4.0))
for ax, (w, ylab, ttl) in zip(axs, (
        (None, r"$\Delta\, dN_{ch}/d\Delta\phi$",
         "Charged-hadron excess vs angle to the jet"),
        ("pt", r"$\Delta\, dp_T/d\Delta\phi$ [GeV]",
         r"$p_T$-weighted: where the deposited energy goes"))):
    d, e = delta(lambda h: h.hist(h.dphi, dphi_edges,
                                  h.charged & (np.abs(h.deta) < MID),
                                  weights=(h.pt if w else None), events=EV))
    ax.axhline(0, color="0.5", lw=0.8)
    ax.errorbar(dpc, d / dw, e / dw, color=C_DIFF, marker="o", ms=4, lw=1.4, capsize=0,
                label="jet − background")
    ax.axvspan(-np.pi / 2, np.pi / 2, color=C_JET, alpha=0.06, lw=0)
    ax.set_xlim(-np.pi, np.pi)
    ax.set_xticks([-np.pi, -np.pi / 2, 0, np.pi / 2, np.pi],
                  [r"$-\pi$", r"$-\pi/2$", "0", r"$\pi/2$", r"$\pi$"])
    tidy(ax, r"$\Delta\phi = \phi - \phi_J$", ylab, ttl + f",  $|\\Delta\\eta|<{MID}$")
for ax in axs:     # labels pinned to the top of the frame, clear of the data
    tr = ax.get_xaxis_transform()
    ax.text(-np.pi / 2 + 0.08, 0.97, "near side", transform=tr, va="top", fontsize=9, color="0.3")
    ax.text(np.pi - 0.08, 0.97, "away", transform=tr, ha="right", va="top", fontsize=9, color="0.3")
fig.tight_layout()

In [ ]:
# the (Delta eta, Delta phi) picture, and how significant each cell is
de_edges = np.linspace(-3, 3, 25)
dp_edges = np.linspace(-np.pi, np.pi, 25)
area = np.outer(np.diff(de_edges), np.diff(dp_edges))
d, e = delta(lambda h: h.hist((h.deta, h.dphi), (de_edges, dp_edges), h.charged,
                              weights=h.pt, events=EV))
fig, axs = plt.subplots(1, 2, figsize=(12.5, 4.4))
ext = [dp_edges[0], dp_edges[-1], de_edges[0], de_edges[-1]]
v = np.nanpercentile(np.abs(d / area), 99)
im = axs[0].imshow(d / area, origin="lower", aspect="auto", extent=ext, cmap=CMAP_DIFF,
                   vmin=-v, vmax=v)
fig.colorbar(im, ax=axs[0], label=r"$\Delta\, d^2p_T/d\Delta\eta\,d\Delta\phi$ [GeV]")
sig = np.divide(d, e, out=np.zeros_like(d), where=e > 0)
im = axs[1].imshow(sig, origin="lower", aspect="auto", extent=ext, cmap=CMAP_DIFF, vmin=-5, vmax=5)
fig.colorbar(im, ax=axs[1], label=r"$\Delta / \sigma$")
for ax, t in zip(axs, ("Charged $p_T$ excess around the jet", "Significance per cell")):
    ax.set_xticks([-np.pi, -np.pi / 2, 0, np.pi / 2, np.pi],
                  [r"$-\pi$", r"$-\pi/2$", "0", r"$\pi/2$", r"$\pi$"])
    ax.set_xlabel(r"$\Delta\phi$"); ax.set_ylabel(r"$\Delta\eta$"); ax.set_title(t, loc="left")
fig.tight_layout()

In [ ]:
# in pT slices: where is the excess soft, where is the depletion?
slices = [(0.0, 0.5), (0.5, 1.0), (1.0, 2.0), (2.0, 4.0)]
fig, axs = plt.subplots(1, len(slices), figsize=(14, 3.6), sharex=True)
for ax, (lo, hi) in zip(axs, slices):
    d, e = delta(lambda h: h.hist(h.dphi, dphi_edges,
                                  h.charged & (np.abs(h.deta) < MID) & (h.pt >= lo) & (h.pt < hi),
                                  events=EV))
    ax.axhline(0, color="0.5", lw=0.8)
    ax.errorbar(dpc, d / dw, e / dw, color=C_DIFF, marker="o", ms=3, lw=1.2, capsize=0)
    ax.set_xlim(-np.pi, np.pi)
    ax.set_xticks([-np.pi, 0, np.pi], [r"$-\pi$", "0", r"$\pi$"])
    tidy(ax, r"$\Delta\phi$", r"$\Delta\, dN_{ch}/d\Delta\phi$" if ax is axs[0] else None,
         f"{lo} < $p_T$ < {hi} GeV")
fig.suptitle(f"The excess and the depletion by $p_T$,  $|\\Delta\\eta|<{MID}$",
             x=0.01, ha="left", fontsize=12)
fig.tight_layout()

In [ ]:
# the spectrum of the excess: near side vs away side, against the background's shape
pt_edges2 = np.linspace(0, 3, 16)
ptc2, pw = centers(pt_edges2), np.diff(pt_edges2)
regions = {"near  $|\\Delta\\phi|<\\pi/2$": lambda h: np.abs(h.dphi) < np.pi / 2,
           "away  $|\\Delta\\phi|>\\pi/2$": lambda h: np.abs(h.dphi) >= np.pi / 2}
fig, axs = plt.subplots(1, 2, figsize=(12.5, 4.0))
for (name, sel), ls, mk in zip(regions.items(), ("-", "--"), ("o", "s")):
    d, e = delta(lambda h: h.hist(h.pt, pt_edges2, h.charged & (np.abs(h.deta) < MID) & sel(h),
                                  events=EV))
    axs[0].errorbar(ptc2, d / pw, e / pw, color=C_DIFF, ls=ls, marker=mk, ms=4, lw=1.3,
                    capsize=0, label=name)
    b, be = H["bg"].hist(H["bg"].pt, pt_edges2,
                         H["bg"].charged & (np.abs(H["bg"].deta) < MID) & sel(H["bg"]), events=EV)
    r = np.divide(d, b, out=np.full_like(d, np.nan), where=b > 0)
    re = np.divide(e, b, out=np.full_like(e, np.nan), where=b > 0)
    axs[1].errorbar(ptc2, 100 * r, 100 * re, color=C_DIFF, ls=ls, marker=mk, ms=4, lw=1.3,
                    capsize=0, label=name)
for ax in axs:
    ax.axhline(0, color="0.5", lw=0.8)
tidy(axs[0], r"$p_T$ [GeV]", r"$\Delta\, dN_{ch}/dp_T$ [GeV$^{-1}$]", "Spectrum of the jet-induced hadrons")
tidy(axs[1], r"$p_T$ [GeV]", "(jet − bg) / bg  [%]", "Relative to the background")
axs[0].legend(frameon=False); axs[1].legend(frameon=False)
fig.tight_layout()

### Characterizing the wake

The numbers the figures above show, per event:

- **yield**, **energy** and $p_T$ **along the jet** of the near-side excess and of the
  away-side depletion;
- the excess's **width** in $\Delta\phi$ and $\Delta\eta$: the RMS of the $p_T$-weighted
  excess about the jet;
- its **mean $p_T$**, against the bulk's, which tells whether it is thermal matter (the wake)
  or something harder;
- its **composition**: $K/\pi$ and $p/\pi$ in the excess against the background. A wake that
  is simply hotter or faster-moving medium carries more baryons per pion.

In [ ]:
near = lambda h: h.charged & (np.abs(h.deta) < MID) & (np.abs(h.dphi) < np.pi / 2)   # noqa: E731
away = lambda h: h.charged & (np.abs(h.deta) < MID) & (np.abs(h.dphi) >= np.pi / 2)  # noqa: E731

def line(label, fn, unit=""):
    d, e = delta(fn)
    print(f"  {label:34s} {d:+9.3f} +- {e:6.3f} {unit:4s} ({abs(d) / e if e else 0:4.1f} sigma)")
    return d, e

print(f"Per event, |Delta eta| < {MID}, {len(EV)} events:\n")
print("near side (|dphi| < pi/2)")
nN = line("N_ch", lambda h: h.total(near(h), events=EV))
nE = line("E", lambda h: h.total(near(h), h.E, events=EV), "GeV")
nP = line("p_T along the jet", lambda h: h.total(near(h), h.p_along, events=EV), "GeV")
print("away side (|dphi| > pi/2)")
aN = line("N_ch", lambda h: h.total(away(h), events=EV))
aE = line("E", lambda h: h.total(away(h), h.E, events=EV), "GeV")
aP = line("p_T along the jet", lambda h: h.total(away(h), h.p_along, events=EV), "GeV")

# width of the near-side excess: RMS of the pT-weighted excess (bins with a net excess)
def rms(values, edges, sel):
    d, _ = delta(lambda h: h.hist(values(h), edges, sel(h), weights=h.pt, events=EV))
    c = centers(edges); w = np.clip(d, 0, None)
    return np.sqrt((w * c ** 2).sum() / w.sum()) if w.sum() > 0 else np.nan
wphi = rms(lambda h: h.dphi, np.linspace(-np.pi / 2, np.pi / 2, 19), near)
weta = rms(lambda h: h.deta, np.linspace(-MID, MID, 11),
           lambda h: h.charged & (np.abs(h.deta) < MID) & (np.abs(h.dphi) < np.pi / 2))
print(f"\nnear-side excess width (pT-weighted RMS): dphi {wphi:.2f} rad, deta {weta:.2f}")

# mean pT of the excess vs the bulk
dS, eS = delta(lambda h: h.total(near(h), h.pt, events=EV))
bN, _ = H["bg"].total(near(H["bg"]), events=EV)
bS, _ = H["bg"].total(near(H["bg"]), H["bg"].pt, events=EV)
mpt = dS / nN[0]
mpt_e = abs(mpt) * np.hypot(eS / dS, nN[1] / nN[0])
print(f"<p_T> of the near-side excess {mpt:.3f} +- {mpt_e:.3f} GeV;  background {bS / bN:.3f} GeV")

print("\ncomposition, near side                excess            background")
for a, b in (("K", "pi"), ("p+pbar", "pi")):
    num = delta(lambda h: h.total(near(h) & h.species(a), events=EV))
    den = delta(lambda h: h.total(near(h) & h.species(b), events=EV))
    r = num[0] / den[0]
    re = abs(r) * np.hypot(num[1] / num[0], den[1] / den[0])
    bn = H["bg"].total(near(H["bg"]) & H["bg"].species(a), events=EV)[0]
    bd = H["bg"].total(near(H["bg"]) & H["bg"].species(b), events=EV)[0]
    print(f"  {a + '/' + b:10s}                     {r:6.3f} +- {re:5.3f}      {bn / bd:6.3f}")

### Reading the result (default run: 4 PGun events × 1000 oversamples)

The jet deposits 13–38 GeV per event (29 GeV on average) into this medium.

- **The wake is identified on the near side**, within $|\Delta\eta| < 1$:
  $+11.6 \pm 0.6$ charged hadrons (20σ) and $+10.5 \pm 0.3$ GeV of $p_T$ along the jet (34σ)
  per event.
- **It is compact.** The $p_T$-weighted RMS is about 0.55 in $\Delta\phi$ and 0.44 in
  $\Delta\eta$: much wider than the jet cone of a 60 GeV parton, much narrower than the
  event.
- **It is harder than the bulk.** $\langle p_T\rangle = 1.05 \pm 0.06$ GeV against 0.62 GeV
  for the background at the same angles. Relative to the background, the excess grows with
  $p_T$. $K/\pi$ is enhanced (0.24 against 0.16); $p/\pi$ is not significantly
  (0.055 ± 0.013 against 0.046). That is the signature of flow-boosted matter: the deposit
  drives a flow along the jet, and Cooper–Frye turns it into hadrons emitted along the jet.
- **The diffusion wake appears on the away side at 2.4σ.** There are $-1.3 \pm 0.6$ charged
  hadrons at $|\Delta\phi| > \pi/2$. Those hadrons carry $+0.7 \pm 0.3$ GeV of $p_T$ *along*
  the jet: fewer hadrons move against it, which is what the diffusion wake predicts. The
  depletion is clearest below 0.5 GeV, near $\Delta\phi \approx \pm 0.8\pi$. §5 shows it
  needs ~4× the statistics for 5σ.

Rerun with `--events`/`--oversample` and these numbers change; the cells above recompute them.

## 4. Energy and momentum balance

The strongest check: the jet put a known four-momentum into the fluid (the source's own
accounting, recorded per event). Summed over **all** hadrons, $\Delta E$ and $\Delta p$ along
the jet must give it back, up to what leaves through the grid's open η edges and iSS's
rapidity window. In a finite window only part of it arrives. This shows how much, as a
function of the window.

$\Delta E$ over all rapidities sits on the bulk's ~$10^4$ GeV, and needs far more statistics
than $\Delta p$ along the jet, where the bulk averages to zero.

In [ ]:
Edep = np.mean([META["jet"]["events"][k]["deposited_P"][0] for k in EV])
Pdep = np.mean([np.dot(META["jet"]["events"][k]["deposited_P"][1:3],
                       [np.cos(AXIS[k][0]), np.sin(AXIS[k][0])]) for k in EV])
windows = np.array([0.25, 0.5, 1.0, 1.5, 2.0, 3.0, 4.0, 5.0])
res = {"E": [], "p": []}
for Yw in windows:
    res["E"].append(delta(lambda h: h.total(np.abs(h.y) < Yw, h.E, events=EV)))
    res["p"].append(delta(lambda h: h.total(np.abs(h.y) < Yw, h.p_along, events=EV)))

fig, axs = plt.subplots(1, 2, figsize=(12.5, 4.0))
for ax, key, dep, lab in ((axs[0], "p", Pdep, r"$\Delta p_T$ along the jet"),
                          (axs[1], "E", Edep, r"$\Delta E$")):
    m, s = np.array(res[key]).T
    ax.axhline(dep, color=C_JET, lw=1.2, ls="--")
    ax.text(windows[0], dep, "  deposited", va="bottom", fontsize=9, color="0.3")
    ax.axhline(0, color="0.5", lw=0.8)
    ax.errorbar(windows, m, s, color=C_DIFF, marker="o", ms=4, lw=1.3, capsize=0)
    tidy(ax, r"rapidity window $|y| < Y$", lab + " [GeV per event]", lab + " in the hadrons")
fig.tight_layout()
m, s = res["p"][-1]
print(f"all rapidities: Delta p_along = {m:.2f} +- {s:.2f} GeV   (deposited {Pdep:.2f})")
m, s = res["E"][-1]
print(f"                Delta E       = {m:.2f} +- {s:.2f} GeV   (deposited {Edep:.2f})")

## 5. How much statistics a wake measurement needs

Every error above falls as $1/\sqrt{N_\text{events} \times N_\text{os}}$: both legs are
Poisson-sampled from a fixed surface, and each event has its own background. This scales
the present errors to show what it takes to see each feature at 5σ.

In [ ]:
N_now = len(EV) * min(H["jet"].n_os, H["bg"].n_os)
feat = {"near-side N_ch": nN, "near-side p_T along jet": nP,
        "away-side N_ch (diffusion wake)": aN, "away-side p_T along jet": aP}
print(f"now: {len(EV)} events x {min(H['jet'].n_os, H['bg'].n_os)} oversamples "
      f"= {N_now} (event x oversample) per leg\n")
print(f"{'feature':34s} {'now':>7s}   {'needed for 5 sigma':>20s}")
for name, (d, e) in feat.items():
    z = abs(d) / e if e else 0
    need = N_now * (5 / z) ** 2 if z > 0 else np.inf
    print(f"  {name:32s} {z:5.1f} sigma   {need:14.0f}  (x{need / N_now:.2g})")

## 6. What this does not include

- **No $\delta f$.** The fluid is viscous, but the Cooper–Frye is ideal ($\pi^{\mu\nu} = \Pi
  = 0$ on the surface). Shear $\delta f$ mostly hardens the spectra above ~1.5 GeV and
  reduces $v_2$. The wake's soft excess should be affected much less than the bulk's tails.
- **No hadronic rescattering.** iSS decays the resonances, but there is no SMASH stage. It
  would redistribute the excess in momentum, and some of it in $\Delta\phi$.
- **Soft hadrons only.** The jet's surviving partons are not hadronized, so near $\Delta\phi
  \approx 0$ at high $p_T$ this is missing the jet itself. That is exactly what makes the soft
  response visible here, and what a measurement has to subtract.
- **The medium is normalized, not tuned.** `initial_state.target_T` = 0.39 reproduces the
  measured multiplicity and the identified yields (§2). Nothing else is fitted: $\langle
  p_T\rangle$ comes out ~10% harder than data, and $v_n$ are not tuned.
- **Open η edges.** The grid ends at $|\eta| = 5$ while still slightly above $T_\text{sw}$
  there. Energy leaving through it is missing from the all-rapidity balance in §4.